In [ ]:
"""
Notebook that makes requests to various endpoints with various parameter combinations (chem_id, fp_id, sel_by, etc.)
Measures time lapsed for each endpoint. Simulates user experience by going through each endpoint in the same order as UI.
Repeats 5 times to average things out/get some consistent values.
"""

import requests
import time
import pandas as pd
from collections import defaultdict
from genraweb.resources import DB
import json
import numpy as np

rows = []

host = "v2626umcth849.rtord.epa.gov"
ports = [31010, 31000]
chem_ids = ["DTXCID606", "DTXCID30182", "OC(C)COC(CO)CO"]
fp_ids = ["chm_mrgn", "chm_httr", "chm_ct", "bio_txct"]
sel_bys = ["tox_txrf", "bio_txct", "no_filter"]
combos = [("tox_txrf", "tox_fp"), ("tox_txrf", "tox_fp_dosage"), ("bio_txct", "bio_fp")]
endpoints = [
    "uiSetup",
    "uiRadialView",
    "uiFastNN",
    "uiPhyschemPlot",
    "uiFingerPrintHeatChart",
    "uiAssayList",
    "uiGenerateReadAcross",
]

# diff deployment types (prev. vs. new)
for port in ports:
    
    for iter_idx in range(5):
        
        completion_list = defaultdict(set)

        clear_cache_url = (
            f"http://{host}"
            f":{port}/api/genra/v3/uiClearCache"  # note the /genra-api prefix
        )

        # diff chem_id
        for chem_id in chem_ids:

            # diff fp_id
            for fp_id in fp_ids:

                # diff sel_by
                for sel_by in sel_bys:

                    # diff summarise/sumrs_by (toxref, pred_type)
                    for summarise, sumrs_by in combos:

                        # before simulate round of UI
                        requests.get(clear_cache_url)

                        # diff endpoint
                        for endpoint in endpoints:

                            # filter exceptions
                            if sel_by == "no_filter":
                                # no_filter isn't complete
                                if endpoint in ["uiAssayList", "uiGenerateReadAcross"]:
                                    # no need to do uiAL and uiGRA
                                    continue
                                if endpoint == "uiFastNN":
                                    # let's skip no_filter for explorer
                                    continue
                            else:
                                # filters and panel 4 selections should match
                                if "tox" in sel_by and "bio" in summarise:
                                    continue
                                elif "bio" in sel_by and "tox" in summarise:
                                    continue

                            # chem_id + fp_id exceptions
                            if "DTX" not in chem_id and "chm" not in fp_id:
                                # custom SMILE with non-chem fp_id won't work
                                continue

                            # endpoint exceptions
                            curr_fp_id = None
                            if endpoint == "uiSetup":
                                if chem_id in completion_list[endpoint]:
                                    # uiSetup only varies by chem_id
                                    continue
                                else:
                                    completion_list[endpoint].add(chem_id)
                            elif endpoint in ["uiRadialView", "uiPhyschemplot", "uiFingerprintHeadChart"]:
                                if (chem_id, fp_id, sel_by) in completion_list[endpoint]:
                                    # these only vary by (chem_id, fp_id, sel_by)
                                    continue
                                else:
                                    completion_list[endpoint].add((chem_id, fp_id, sel_by))
                            elif endpoint == "uiFastNN":
                                if (chem_id, sel_by) in completion_list[endpoint]:
                                    # uiFastNN only varies by (chem_id, sel_by)
                                    continue
                                else:
                                    completion_list[endpoint].add((chem_id, sel_by))
                                    # adjust fp_id for uiFastNN so that it has a mix of fp_ids
                                    curr_fp_id = fp_id
                                    fp_id = ",".join(fp_ids)                    

                            url = (
                                f"http://{host}"
                                f":{port}/api/genra/v4/"
                                f"{endpoint}?s0=0.1&k0=10&graph_type=all_nhgbrs&steps=3&"
                                f"chem_id={chem_id}&fp={fp_id}&"
                                f"sel_by={sel_by}&summarise={summarise}&"
                                f"sumrs_by={sumrs_by}"
                            )

                            now = time.time()
                            res = requests.get(url)
                            rt = time.time() - now

                            row = {
                                "port": port,
                                "endpoint": endpoint,
                                "chem_id": chem_id,
                                "fp_id": fp_id,
                                "sel_by": sel_by,
                                "summarise": summarise,
                                "sumrs_by": sumrs_by,
                                "rt": rt,
                                "content": res.text[:100],
                                "status_code": res.status_code,
                                "iter_idx": iter_idx,
                            }
                            rows.append(row)

                            if curr_fp_id is not None:
                                fp_id = curr_fp_id

df = pd.DataFrame(rows)
with open("performance_data.json", "w") as f:
    json.dump(rows, f)
    
## some basics to look at
print(np.mean(df.rt), np.mean(df[df.rt > 1.0].rt), len(df[df.rt > 1.0]))
df